# Return Period Analysis Storm Hans

## 0.5° Resolution/ERA5 Analysis/2day Precip.: Time-Series of Weighted Catchment Daily Precipitation and Return Period Analysis

### Dataset/resolution choices

In [1]:
# %% [Setup & dataset selection]
# ─────────────────────────────────────────────────────────────────────────────
# CHANGE ONLY THIS CELL to switch dataset, resolution, or accumulation window.
# Everything else in the notebook runs identically regardless of what you set here.
# ─────────────────────────────────────────────────────────────────────────────

import sys
from pathlib import Path

HELPER_DIR = Path("/nird/home/lbal/helper")
if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))

# ── Dataset selection ─────────────────────────────────────────────────────────
DATASET    = "era5"       # Switch to "senorge" for Senorge runs
RESOLUTION = "0.5x0.5"   # Switch to "0.25x0.25" for higher-res ERA5
                          # Use "" (empty string) for Senorge (no resolution suffix)

# ── Accumulation window ───────────────────────────────────────────────────────
# 1 = daily (1-day) precipitation
# 2 = 2-day rolling accumulation
WINDOW_DAYS = 2

# ── Recompute flag ────────────────────────────────────────────────────────────
# False → use cached catchment NetCDF if it exists (fast, seconds)
# True  → recompute from raw ERA5 files (slow, several minutes)
# After the first successful run, leave this as False.
FORCE_RECOMPUTE = True

# ── Figure subfolder ─────────────────────────────────────────────────────────
# PDFs are saved to:  /nird/datalake/NS9873K/lbal/figures/<FIG_SUBDIR>/
# Change FIG_SUBDIR in future notebooks to separate outputs by project.
FIG_SUBDIR = "timeseries_return_hans"

print(f"Dataset    : {DATASET}")
print(f"Resolution : {RESOLUTION}")
print(f"Window     : {WINDOW_DAYS}-day")
print(f"Recompute  : {FORCE_RECOMPUTE}")
print(f"Fig subdir : {FIG_SUBDIR}")

Dataset    : era5
Resolution : 0.5x0.5
Window     : 2-day
Recompute  : True
Fig subdir : timeseries_return_hans


### Configuration check

Verify that all paths resolve correctly before running the full analysis

In [2]:
# %% [Configuration check]
import config_paths as cfg

tag = cfg.res_tag(DATASET, RESOLUTION)

print(f"Resolution tag  : {tag}")
print(f"ERA5 raw dir    : {cfg.ERA5_RAW_DIR}")
print(f"Catchment dir   : {cfg.CATCHMENT_RAW_DIR}")
print(f"Postproc dir    : {cfg.postproc_dir(DATASET, RESOLUTION)}")
print(f"Figures dir     : {cfg.FIGURES_DIR / FIG_SUBDIR}")
print(f"Hans year       : {cfg.HANS_SEARCH_YEAR}")
print()
print("Catchments:")
for slug, title in cfg.CATCHMENTS.items():
    nc = cfg.catchment_nc_path(DATASET, RESOLUTION, slug)
    cached = "✓ cached" if nc.exists() else "  (not yet cached)"
    print(f"  {slug:30s}  {cached}")

Resolution tag  : era5_0.5x0.5
ERA5 raw dir    : /nird/datapeak/NS9873K/etdu/raw/era5/continuous-format/europe/daily/tp24
Catchment dir   : /nird/datalake/NS9873K/etdu/raw/nve
Postproc dir    : /nird/datalake/NS9873K/lbal/postprocessed/era5_0.5x0.5
Figures dir     : /nird/datalake/NS9873K/lbal/figures/timeseries_return_hans
Hans year       : 2023

Catchments:
  nevina_bergheim                 ✓ cached
  nevina_honnefoss                ✓ cached
  nevina_losna                    ✓ cached
  regine_drammen                  ✓ cached
  regine_glomma                   ✓ cached


### Run full analysis on all catchments

Loops over all five catchments automatically.  
For each catchment:
1. Loads or computes the weighted daily time series (cached to `postprocessed/`)
2. Fits GEV to annual maxima
3. Saves a two-panel PDF to `figures/timeseries_return_hans/`

**First run:** `FORCE_RECOMPUTE = False` still works — cache is missing so it computes automatically.  
**Subsequent runs:** cached NetCDFs are reused, only the figures are regenerated.

In [3]:
# %% [Run full analysis]
from catchment_tools import run_all

# Alle catchments in for-loop runnen
run_all(
    dataset         = DATASET,
    resolution      = RESOLUTION,
    window_days     = WINDOW_DAYS,
    force_recompute = FORCE_RECOMPUTE,
    fig_subdir      = FIG_SUBDIR,
)


[run_all] Dataset: era5 | Resolution: 0.5x0.5
[run_all] ERA5 files found: 84  (1941–2024)
  Opening 84 ERA5 files (lazy) ...


/nird/home/lbal/helper/data_era5.py:51: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset([str(f) for f in era5_files], combine="by_coords", coords="minimal", compat="override", chunks={"time": 365},)
/nird/home/lbal/helper/data_era5.py:51: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset([str(f) for f in era5_files], combine="by_coords", coords="minimal", compat="override", chunks={"time": 365},)
/nird/home/lbal/helper/data_era5.py:51: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset([str(f) for f in era5_files], comb


── Catchment: Nevina Bergheim (nevina_bergheim) ──
  Computing weighted mean (this triggers Dask computation) ...
    [cache] Saved → tp_catchment_nevina_bergheim_era5_0.5x0.5.nc
    [fig]   Saved → timeseries_returnperiod_hans_era5_0.5x0.5_2day_nevina_bergheim_dailyprecip_1941-2024.pdf

── Catchment: Nevina Hønnefoss (nevina_honnefoss) ──
  Computing weighted mean (this triggers Dask computation) ...
    [cache] Saved → tp_catchment_nevina_honnefoss_era5_0.5x0.5.nc
    [fig]   Saved → timeseries_returnperiod_hans_era5_0.5x0.5_2day_nevina_honnefoss_dailyprecip_1941-2024.pdf

── Catchment: Nevina Losna (nevina_losna) ──
  Computing weighted mean (this triggers Dask computation) ...
    [cache] Saved → tp_catchment_nevina_losna_era5_0.5x0.5.nc
    [fig]   Saved → timeseries_returnperiod_hans_era5_0.5x0.5_2day_nevina_losna_dailyprecip_1941-2024.pdf

── Catchment: Regine Drammen (regine_drammen) ──
  Computing weighted mean (this triggers Dask computation) ...
    [cache] Saved → tp_catch

## 0.5° Resolution/ERA5 Analysis/1-day Precip.: Time-Series of Weighted Catchment Daily Precipitation and Return Period Analysis

### Dataset/resolution choices

In [4]:
# %% [Setup & dataset selection]
# ─────────────────────────────────────────────────────────────────────────────
# CHANGE ONLY THIS CELL to switch dataset, resolution, or accumulation window.
# Everything else in the notebook runs identically regardless of what you set here.
# ─────────────────────────────────────────────────────────────────────────────

import sys
from pathlib import Path

HELPER_DIR = Path("/nird/home/lbal/helper")
if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))

# ── Dataset selection ─────────────────────────────────────────────────────────
DATASET    = "era5"       # Switch to "senorge" for Senorge runs
RESOLUTION = "0.5x0.5"   # Switch to "0.25x0.25" for higher-res ERA5
                          # Use "" (empty string) for Senorge (no resolution suffix)

# ── Accumulation window ───────────────────────────────────────────────────────
# 1 = daily (1-day) precipitation
# 2 = 2-day rolling accumulation
WINDOW_DAYS = 1

# ── Recompute flag ────────────────────────────────────────────────────────────
# False → use cached catchment NetCDF if it exists (fast, seconds)
# True  → recompute from raw ERA5 files (slow, several minutes)
# After the first successful run, leave this as False.
FORCE_RECOMPUTE = True

# ── Figure subfolder ─────────────────────────────────────────────────────────
# PDFs are saved to:  /nird/datalake/NS9873K/lbal/figures/<FIG_SUBDIR>/
# Change FIG_SUBDIR in future notebooks to separate outputs by project.
FIG_SUBDIR = "timeseries_return_hans"

print(f"Dataset    : {DATASET}")
print(f"Resolution : {RESOLUTION}")
print(f"Window     : {WINDOW_DAYS}-day")
print(f"Recompute  : {FORCE_RECOMPUTE}")
print(f"Fig subdir : {FIG_SUBDIR}")

Dataset    : era5
Resolution : 0.5x0.5
Window     : 1-day
Recompute  : True
Fig subdir : timeseries_return_hans


### Run full analysis on all catchments

In [5]:
# %% [Run full analysis]
from catchment_tools import run_all

# Alle catchments in for-loop runnen
run_all(
    dataset         = DATASET,
    resolution      = RESOLUTION,
    window_days     = WINDOW_DAYS,
    force_recompute = FORCE_RECOMPUTE,
    fig_subdir      = FIG_SUBDIR,
)


[run_all] Dataset: era5 | Resolution: 0.5x0.5
[run_all] ERA5 files found: 84  (1941–2024)
  Opening 84 ERA5 files (lazy) ...


/nird/home/lbal/helper/data_era5.py:51: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset([str(f) for f in era5_files], combine="by_coords", coords="minimal", compat="override", chunks={"time": 365},)
/nird/home/lbal/helper/data_era5.py:51: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset([str(f) for f in era5_files], combine="by_coords", coords="minimal", compat="override", chunks={"time": 365},)
/nird/home/lbal/helper/data_era5.py:51: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset([str(f) for f in era5_files], comb


── Catchment: Nevina Bergheim (nevina_bergheim) ──
  Computing weighted mean (this triggers Dask computation) ...
    [cache] Saved → tp_catchment_nevina_bergheim_era5_0.5x0.5.nc
    [fig]   Saved → timeseries_returnperiod_hans_era5_0.5x0.5_1day_nevina_bergheim_dailyprecip_1941-2024.pdf

── Catchment: Nevina Hønnefoss (nevina_honnefoss) ──
  Computing weighted mean (this triggers Dask computation) ...
    [cache] Saved → tp_catchment_nevina_honnefoss_era5_0.5x0.5.nc
    [fig]   Saved → timeseries_returnperiod_hans_era5_0.5x0.5_1day_nevina_honnefoss_dailyprecip_1941-2024.pdf

── Catchment: Nevina Losna (nevina_losna) ──
  Computing weighted mean (this triggers Dask computation) ...
    [cache] Saved → tp_catchment_nevina_losna_era5_0.5x0.5.nc
    [fig]   Saved → timeseries_returnperiod_hans_era5_0.5x0.5_1day_nevina_losna_dailyprecip_1941-2024.pdf

── Catchment: Regine Drammen (regine_drammen) ──
  Computing weighted mean (this triggers Dask computation) ...
    [cache] Saved → tp_catch

## 0.25° Resolution/ERA5 Analysis/2-day Precip.: Time-Series of Weighted Catchment Daily Precipitation and Return Period Analysis

### Dataset/resolution choices

### Run full analysis on all catchments

## 0.25° Resolution/ERA5 Analysis/1-day Precip.: Time-Series of Weighted Catchment Daily Precipitation and Return Period Analysis

### Dataset/resolution choices

### Run full analysis on all catchments

## 1x1km Resolution/Senorge/2-day Precip.: Time-Series of Weighted Catchment Daily Precipitation and Return Period Analysis

### Dataset/resolution choices

### Run full analysis on all catchments

## 1x1km Resolution/Senorge/1-day Precip.: Time-Series of Weighted Catchment Daily Precipitation and Return Period Analysis

### Dataset/resolution choices

### Run full analysis on all catchments